# 01. Jev 응답 형식·확신도 확인

**목적:** 판단 엔진 설계 전에 Jev의 실제 응답 구조와 확신도 분포를 확인한다.

확인할 것:
1. Choice / Score / Noul 응답 JSON 구조
2. 확신도 형태와 답안 품질별 분포
3. 질문 수 증가에 따른 지연 시간
4. 한국어 서술형 답안 판단 품질

- 모델: `jev-1.13.0` (버전 고정), 비교용 `jev-1.14.0`
- 날짜: 2026-10-15 ~ 2026-11-03

In [ ]:
import json, os, time
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

URL = "https://api.typesafe.ai/v1/systemone"
MODEL = "jev-1.13.0"
OUT_DIR = Path("outputs/01_jev_response_check")
OUT_DIR.mkdir(parents=True, exist_ok=True)

client = httpx.Client(
    headers={"Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}"},
    timeout=30,
)


## 문제와 채점 기준


In [ ]:
PROBLEM = "신경망 학습에서 역전파(backpropagation)가 어떻게 동작하는지 설명하시오."

QUESTIONS = {
    "mentions_chain_rule": {
        "type": "noul",
        "instructions": "The `answer` explains that backpropagation uses the chain rule to compute gradients.",
        "criteria": {
            "true": "Explicitly mentions the chain rule, or multiplying local derivatives layer by layer",
            "false": "No mention of the chain rule or composing derivatives",
        },
    },
    "mentions_weight_update": {
        "type": "noul",
        "instructions": "The `answer` states that the computed gradients are used to update the weights.",
    },
    "accuracy": {
        "type": "score",
        "instructions": "How accurate is the `answer`'s explanation of how backpropagation works?",
        "criteria": [
            "Wrong, or not about backpropagation at all",
            "Mentions errors or gradients flowing backward, but the mechanism is missing or incorrect",
            "Mechanism mostly correct, with one notable error or omission",
            "Correct and complete: forward pass, loss, gradients via chain rule, weight update",
        ],
    },
    "misconception": {
        "type": "choice",
        "instructions": "Which misconception, if any, appears in the `answer`?",
        "criteria": {
            "none": "No misconception about backpropagation",
            "is_optimizer": "Treats backpropagation itself as the optimizer that updates weights, conflating it with gradient descent",
            "error_copied": "Claims the same error value is passed unchanged to every layer, without derivatives",
            "reverse_inference": "Describes backpropagation as running the network in reverse to recompute inputs",
            "other": None,
        },
    },
}


## 테스트 답안
의도: 모범 / 부분 정답 / 오개념 / 무관 / 빈약 / 장황하지만 핵심 누락


In [ ]:
ANSWERS = {
    "model": (
        "역전파는 먼저 순전파로 출력과 손실을 계산한 뒤, 출력층에서 입력층 방향으로 "
        "손실에 대한 각 가중치의 기울기를 구하는 방법이다. 이때 연쇄법칙을 이용해 각 층의 "
        "국소 미분을 곱해 나가며 기울기를 효율적으로 계산한다. 구한 기울기는 경사하강법 등 "
        "최적화 알고리즘이 가중치를 갱신하는 데 사용된다."
    ),
    "partial": (
        "출력에서 생긴 오차를 이용해 뒤쪽 층부터 앞쪽 층으로 기울기를 계산하고, "
        "그 기울기 방향의 반대로 가중치를 조금씩 바꿔서 손실을 줄인다."
    ),
    "misconception": (
        "역전파는 가중치를 직접 최적화하는 알고리즘이다. 출력층에서 계산한 오차 값을 "
        "그대로 모든 층에 똑같이 전달해서 각 층이 그 오차만큼 가중치를 고친다."
    ),
    "irrelevant": (
        "CNN에서는 합성곱 층으로 특징을 추출하고 풀링 층으로 크기를 줄인다. "
        "마지막에 완전연결층을 통해 분류를 수행한다."
    ),
    "thin": "오차를 뒤로 보내서 학습한다.",
    "verbose": (
        "역전파는 딥러닝에서 매우 중요한 개념으로, 신경망이 학습할 수 있게 해 주는 핵심 원리이다. "
        "신경망은 여러 층으로 구성되어 있으며 각 층은 뉴런과 가중치로 이루어진다. 학습 과정에서 "
        "네트워크는 데이터를 보고 점점 더 좋은 결과를 내도록 스스로 조정된다. 역전파 덕분에 "
        "오늘날의 이미지 인식, 번역 등 다양한 AI 기술이 가능해졌으며 매우 효율적인 방법으로 알려져 있다."
    ),
}

def make_state(answer: str) -> dict:
    return {"question": PROBLEM, "answer": answer}

## 호출 함수


In [ ]:
def call_jev(state, questions, tag, model=MODEL, save=True, max_retries=3):
    body = {"state": state, "model": model, "questions": questions}
    for attempt in range(max_retries + 1):
        t0 = time.perf_counter()
        r = client.post(URL, json=body)
        latency = time.perf_counter() - t0
        if r.status_code in (429, 529) and attempt < max_retries:
            time.sleep(float(r.headers.get("retry-after", 2 ** attempt)))
            continue
        if r.is_error:
            print(r.status_code, r.text)
        r.raise_for_status()
        break
    data = r.json()
    if save:
        (OUT_DIR / f"{tag}.json").write_text(json.dumps(data, ensure_ascii=False, indent=2))
    return data, latency


## 단건 원본 확인
파서를 짜기 전에 실제 JSON을 눈으로 확인한다.


In [ ]:
data, latency = call_jev(make_state(ANSWERS["model"]), QUESTIONS, tag="single_model")
print(f"latency: {latency:.3f}s")
print(json.dumps(data, ensure_ascii=False, indent=2))


## 파서
Noul은 confidence가 없으므로 `|2p - 1|`로 정의 (0.5면 0, 0이나 1이면 1).


In [ ]:
def parse_answers(data: dict) -> list[dict]:
    rows = []
    for qid, a in data["answers"].items():
        if a["type"] == "noul":
            p = a["noul"]
            rows.append({"question": qid, "type": "noul", "raw": p, "pred": p >= 0.5, "conf": round(abs(2 * p - 1), 2)})
        elif a["type"] == "score":
            top = max(a["probabilities"], key=a["probabilities"].get)
            rows.append({"question": qid, "type": "score", "raw": a["score"], "pred": int(top), "conf": a["confidence"]})
        elif a["type"] == "choice":
            rows.append({"question": qid, "type": "choice", "raw": a["probabilities"], "pred": a["choice"], "conf": a["confidence"]})
    return rows

pd.DataFrame(parse_answers(data))

## 전체 실행 (답안 6개 × 질문 4개)


In [ ]:
rows = []
for name, text in ANSWERS.items():
    data, latency = call_jev(make_state(text), QUESTIONS, tag=f"run_{name}")
    for row in parse_answers(data):
        rows.append({"answer": name, "latency": latency, "input_tokens": data["usage"]["input_tokens"], **row})

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "run.csv", index=False)


In [ ]:
display(df.pivot(index="answer", columns="question", values="pred"))
display(df.pivot(index="answer", columns="question", values="conf").round(2))
display(df.groupby("answer")[["latency", "input_tokens"]].first())


## V2 질문 세트

V1 결과에서 확인한 설계 문제 반영:

1. **오개념 동시 존재:** Choice는 하나만 고르므로 `misconception` 답안의 두 오개념이 0.78 / 0.21로 갈림 → 오개념별 Noul로 분리
2. **`other`가 "다른 오개념"과 "무관한 답"을 함께 흡수** → `is_on_topic` 게이트 Noul 추가, 무관하면 나머지 항목은 코드에서 skip
3. **Score 0수준이 "틀림"과 "주제 무관"을 섞음** → 주제 무관은 게이트가 맡고, 0수준은 "주제는 맞지만 설명 없음"으로 재정의

`mentions_chain_rule`, `mentions_weight_update`는 V1과 동일하게 두어 비교 기준으로 사용.

In [ ]:
QUESTIONS_V2 = {
    "is_on_topic": {
        "type": "noul",
        "instructions": "The `answer` attempts to explain backpropagation, the topic of the `question`.",
        "criteria": {
            "true": "Talks about backpropagation or how a neural network learns from its errors, even if vague or wrong",
            "false": "Talks about a different topic and never addresses backpropagation",
        },
    },
    "mentions_chain_rule": QUESTIONS["mentions_chain_rule"],
    "mentions_weight_update": QUESTIONS["mentions_weight_update"],
    "accuracy": {
        "type": "score",
        "instructions": "How accurate is the `answer`'s explanation of how backpropagation works?",
        "criteria": [
            "Explains nothing about how it works: only general statements about its importance or about neural networks",
            "Says errors or gradients flow backward, but the mechanism is missing or incorrect",
            "Mechanism mostly correct, with one notable error or omission",
            "Correct and complete: forward pass, loss, gradients via chain rule, weight update",
        ],
    },
    "misc_is_optimizer": {
        "type": "noul",
        "instructions": "The `answer` claims that backpropagation itself is the optimization algorithm that updates the weights.",
        "criteria": {
            "true": "Calls backpropagation the optimizer or weight-update algorithm, conflating it with gradient descent",
            "false": "Treats backpropagation as computing gradients, or describes gradient computation and weight update as separate steps",
        },
    },
    "misc_error_copied": {
        "type": "noul",
        "instructions": "The `answer` claims the same error value is passed unchanged to every layer, without computing derivatives.",
    },
    "misc_reverse_inference": {
        "type": "noul",
        "instructions": "The `answer` describes backpropagation as running the network in reverse to recompute the inputs.",
    },
}

MISC_KEYS = ["misc_is_optimizer", "misc_error_copied", "misc_reverse_inference"]

# 정답 라벨 (noul: True/False, accuracy: 0~3, 애매하면 None + 주석)
# Claude 초안: 질문 instructions/criteria 문구 기준으로 매김. 검토 후 채점 의도와 다르면 수정
EXPECTED_V2 = {
    "model": {
        "is_on_topic": True,
        "mentions_chain_rule": True,
        "mentions_weight_update": True,
        "accuracy": 3,
        "misc_is_optimizer": False,  # 최적화 알고리즘이 기울기로 갱신한다고 구분해서 서술
        "misc_error_copied": False,
        "misc_reverse_inference": False,
    },
    "partial": {
        "is_on_topic": True,
        "mentions_chain_rule": False,  # 층 방향 기울기 계산만 있고 연쇄법칙·미분 곱 언급 없음
        "mentions_weight_update": True,
        "accuracy": 2,  # 경계: 메커니즘 흐름은 맞고 연쇄법칙이 빠짐 (1로 볼 여지 있음)
        "misc_is_optimizer": False,  # 기울기 계산 → 가중치 갱신을 별도 단계로 서술 (criteria false 조건)
        "misc_error_copied": False,
        "misc_reverse_inference": False,
    },
    "misconception": {
        "is_on_topic": True,
        "mentions_chain_rule": False,
        "mentions_weight_update": False,  # 경계: 가중치를 고친다고는 했지만 "기울기로" 갱신은 아님 (문구 기준 False)
        "accuracy": 1,  # 오차가 뒤로 전달된다는 언급은 있으나 메커니즘이 틀림
        "misc_is_optimizer": True,  # "가중치를 직접 최적화하는 알고리즘"
        "misc_error_copied": True,  # "오차 값을 그대로 모든 층에 똑같이 전달"
        "misc_reverse_inference": False,
    },
    "irrelevant": {"is_on_topic": False},  # 주제 무관이면 나머지는 skip
    "thin": {
        "is_on_topic": True,
        "mentions_chain_rule": False,
        "mentions_weight_update": False,  # 기울기·가중치 언급 없음
        "accuracy": 1,  # 경계: "오차를 뒤로 보낸다"는 1수준 묘사와 일치 (0으로 볼 여지 있음)
        "misc_is_optimizer": False,
        "misc_error_copied": False,  # 뒤로 보낸다고만 했고 "그대로 모든 층에" 주장은 없음
        "misc_reverse_inference": False,
    },
    "verbose": {
        "is_on_topic": True,  # 역전파에 대해 말하고 있음 (내용은 없음)
        "mentions_chain_rule": False,
        "mentions_weight_update": False,  # "스스로 조정된다"는 기울기 기반 갱신 서술이 아님
        "accuracy": 0,  # 중요성·일반론만 있음 (0수준 묘사와 일치)
        "misc_is_optimizer": False,
        "misc_error_copied": False,
        "misc_reverse_inference": False,
    },
}

### 게이트
`is_on_topic`이 False면 나머지 항목은 `skipped`. Jev는 한 번에 전부 평가하고, 분기는 코드에서 처리 (판단은 모델, 규칙은 코드).

In [ ]:
GATE = "is_on_topic"

def apply_gate(rows: list[dict]) -> list[dict]:
    on_topic = next(r["pred"] for r in rows if r["question"] == GATE)
    return [{**r, "skipped": not on_topic and r["question"] != GATE} for r in rows]

### V2 전체 실행

In [ ]:
rows_v2 = []
for name, text in ANSWERS.items():
    data, latency = call_jev(make_state(text), QUESTIONS_V2, tag=f"run_v2_{name}")
    for row in apply_gate(parse_answers(data)):
        rows_v2.append({"answer": name, "latency": latency, "input_tokens": data["usage"]["input_tokens"], **row})

df_v2 = pd.DataFrame(rows_v2)
df_v2.to_csv(OUT_DIR / "run_v2.csv", index=False)

In [ ]:
display(df_v2.assign(pred=df_v2.pred.where(~df_v2.skipped, "skip")).pivot(index="answer", columns="question", values="pred"))
display(df_v2.pivot(index="answer", columns="question", values="conf"))
display(df_v2.groupby("answer")[["latency", "input_tokens"]].first())

## V1 vs V2 비교
- accuracy: 0수준 재정의 후 `verbose` 확신도 변화
- 오개념: Choice 1개 → Noul 여러 개로 `misconception` 답안의 두 오개념이 모두 잡히는지
- 자동 처리율: 임계값별 `conf >= t` 비율 (skip 항목 제외)

In [ ]:
acc_v1 = df[df.question == "accuracy"].set_index("answer")[["pred", "conf"]].add_prefix("v1_")
acc_v2 = df_v2[df_v2.question == "accuracy"].set_index("answer")[["pred", "conf", "skipped"]].add_prefix("v2_")
display(acc_v1.join(acc_v2))

misc_v1 = df[df.question == "misconception"].set_index("answer")[["pred", "conf"]].add_prefix("v1_")
misc_v2 = (
    df_v2[df_v2.question.isin(MISC_KEYS) & df_v2.pred.eq(True) & ~df_v2.skipped]
    .groupby("answer")["question"].apply(list)
    .rename("v2_flags")
)
misc_v2_conf = (
    df_v2[df_v2.question.isin(MISC_KEYS)]
    .pivot(index="answer", columns="question", values="conf")
    .add_suffix("_conf")
)
display(misc_v1.join(misc_v2).join(misc_v2_conf).fillna({"v2_flags": ""}))

In [ ]:
def auto_rate(d: pd.DataFrame, thresholds=(0.7, 0.8, 0.9)) -> dict:
    if "skipped" in d:
        d = d[~d.skipped]
    return {t: round((d.conf >= t).mean(), 2) for t in thresholds}

pd.DataFrame({"v1": auto_rate(df), "v2": auto_rate(df_v2)}).rename_axis("threshold")

## 라벨 비교
`EXPECTED_V2`와 Jev 판단 비교. 가장 중요한 것은 **고확신 오답** (확신도는 높은데 라벨과 다른 판단).

In [ ]:
exp = pd.DataFrame(
    [{"answer": a, "question": q, "expected": v} for a, qs in EXPECTED_V2.items() for q, v in qs.items()]
)
cmp = df_v2.merge(exp, on=["answer", "question"])
cmp = cmp[~cmp.skipped & cmp.expected.notna()].copy()
cmp["correct"] = cmp.pred == cmp.expected

if cmp.empty:
    print("EXPECTED_V2가 비어 있음: 라벨을 채운 뒤 다시 실행")
else:
    print(f"일치율: {cmp.correct.mean():.0%} ({cmp.correct.sum()}/{len(cmp)})")
    display(cmp[~cmp.correct][["answer", "question", "raw", "pred", "expected", "conf"]].sort_values("conf", ascending=False))
    cmp["conf_bin"] = pd.cut(cmp.conf, [0, 0.5, 0.8, 0.9, 1.0], include_lowest=True)
    display(cmp.groupby("conf_bin", observed=True).correct.agg(["mean", "count"]))

## 안정성
같은 입력을 반복 호출했을 때 판단과 확신도가 얼마나 흔들리는지. V1/V2 실행 사이에 같은 입력의 확신도가 달라진 것을 확인했음.
V2 질문 × 답안 6개 × 5회 = 30회 호출.

In [ ]:
REPEATS = 5

stab_rows = []
for k in range(REPEATS):
    for name, text in ANSWERS.items():
        data, _ = call_jev(make_state(text), QUESTIONS_V2, tag=f"stab_{name}_{k}", save=False)
        for row in parse_answers(data):
            stab_rows.append({"run": k, "answer": name, **row})

stab = pd.DataFrame(stab_rows)
stab.to_csv(OUT_DIR / "stability.csv", index=False)

stab_summary = (
    stab.assign(pred_str=stab.pred.astype(str))
    .groupby(["answer", "question"])
    .agg(conf_mean=("conf", "mean"), conf_std=("conf", "std"), conf_min=("conf", "min"), n_pred=("pred_str", "nunique"))
    .round(3)
)
stab_summary["flipped"] = stab_summary.n_pred > 1

print(f"판단이 바뀐 칸: {stab_summary.flipped.sum()}/{len(stab_summary)}")
print(f"conf 표준편차 중앙값 {stab_summary.conf_std.median():.3f}, 최대 {stab_summary.conf_std.max():.3f}")
display(stab_summary[stab_summary.flipped | (stab_summary.conf_std >= 0.05)])

## 임계값
Noul 확신도 `|2p-1|`은 Score/Choice보다 엄격 (p=0.11~0.17의 명백한 False도 0.66~0.78). 그래서 임계값을 타입별로 분리.

`|2p-1| >= t`는 `p <= (1-t)/2` 또는 `p >= (1+t)/2`와 같음. 예: t=0.6 → p ≤ 0.2 또는 p ≥ 0.8.

표 값: 자동 처리율 (skip 제외). 라벨이 있으면 자동 처리 항목의 정확도도 출력.

In [ ]:
def is_auto(d: pd.DataFrame, t_noul: float, t_other: float) -> pd.Series:
    return d.conf >= d.type.map(lambda t: t_noul if t == "noul" else t_other)

active = df_v2[~df_v2.skipped]
grid = pd.DataFrame(
    {t_other: {t_noul: round(is_auto(active, t_noul, t_other).mean(), 2) for t_noul in (0.6, 0.7, 0.8, 0.9)}
     for t_other in (0.7, 0.8, 0.9)}
).rename_axis(index="t_noul", columns="t_other")
display(grid)

T_NOUL, T_OTHER = 0.6, 0.8  # 후보값, 결과 보고 조정

if not cmp.empty:
    auto = cmp[is_auto(cmp, T_NOUL, T_OTHER)]
    print(f"T_NOUL={T_NOUL}, T_OTHER={T_OTHER}: 자동 처리 {len(auto)}/{len(cmp)}칸, 그중 정확도 {auto.correct.mean():.0%}")
    display(auto[~auto.correct][["answer", "question", "pred", "expected", "conf"]])

## 지연
문서 주장: 질문 수를 늘려도 응답 시간은 거의 그대로. 질문을 복제해 1 / 4 / 16개로 늘려 3회씩 측정.

In [ ]:
def scaled_questions(n: int) -> dict:
    items = list(QUESTIONS_V2.items()) * (n // len(QUESTIONS_V2) + 1)
    return {f"{qid}_{i}": q for i, (qid, q) in enumerate(items[:n])}

state = make_state(ANSWERS["model"])
call_jev(state, scaled_questions(1), tag="warmup", save=False)

lat_rows = []
for n in (1, 4, 16):
    for k in range(3):
        data, latency = call_jev(state, scaled_questions(n), tag=f"lat_{n}_{k}", save=False)
        lat_rows.append({"n_questions": n, "latency": latency, "input_tokens": data["usage"]["input_tokens"]})

lat_df = pd.DataFrame(lat_rows)
lat_df.groupby("n_questions").agg(
    latency_median=("latency", "median"), latency_max=("latency", "max"), input_tokens=("input_tokens", "first")
).round(3)

## 모델 버전 비교
V2 질문을 `jev-1.14.0`으로 실행해 `jev-1.13.0` 결과(`df_v2`)와 비교. 차이는 안정성 섹션의 실행 간 편차와 함께 해석.

In [ ]:
rows_114 = []
for name, text in ANSWERS.items():
    data, _ = call_jev(make_state(text), QUESTIONS_V2, tag=f"v114_{name}", model="jev-1.14.0")
    for row in apply_gate(parse_answers(data)):
        rows_114.append({"answer": name, **row})

df_114 = pd.DataFrame(rows_114)
ver = df_v2.merge(df_114, on=["answer", "question"], suffixes=("_113", "_114"))
ver["pred_changed"] = ver.pred_113.astype(str) != ver.pred_114.astype(str)
ver["conf_diff"] = (ver.conf_114 - ver.conf_113).round(2)

print(f"판단 변경: {ver.pred_changed.sum()}/{len(ver)}칸")
display(ver[ver.pred_changed | (ver.conf_diff.abs() >= 0.1)][["answer", "question", "pred_113", "pred_114", "conf_113", "conf_114"]])

## 한국어 질문
앱에서는 교수가 한국어로 채점 기준을 입력하므로, 질문을 한국어로 써도 영어 질문(V2)만큼 믿을 수 있는지 확인.

- `QUESTIONS_V2_KO`: V2와 같은 id, 타입, 수준 수. `instructions`/`criteria`만 한국어로 번역
- 의미가 같으므로 라벨은 `EXPECTED_V2`를 그대로 사용
- 비교: 라벨 일치, 임계값(`T_NOUL`, `T_OTHER`) 기준 자동 처리율과 정확도, 칸별 판단·확신도 차이, 안정성
- 호출: 6회 + 안정성 30회

In [ ]:
QUESTIONS_V2_KO = {
    "is_on_topic": {
        "type": "noul",
        "instructions": "`answer`가 `question`의 주제인 역전파를 설명하려고 시도한다.",
        "criteria": {
            "true": "역전파 또는 신경망이 오차로부터 학습하는 방식에 대해 말한다. 모호하거나 틀려도 해당",
            "false": "다른 주제를 다루며 역전파를 전혀 언급하지 않는다",
        },
    },
    "mentions_chain_rule": {
        "type": "noul",
        "instructions": "`answer`가 역전파는 연쇄법칙을 이용해 기울기를 계산한다고 설명한다.",
        "criteria": {
            "true": "연쇄법칙을 명시하거나, 층별 국소 미분을 곱해 나간다고 설명한다",
            "false": "연쇄법칙이나 미분의 합성에 대한 언급이 없다",
        },
    },
    "mentions_weight_update": {
        "type": "noul",
        "instructions": "`answer`가 계산된 기울기를 가중치 갱신에 사용한다고 서술한다.",
    },
    "accuracy": {
        "type": "score",
        "instructions": "`answer`의 역전파 동작 원리 설명은 얼마나 정확한가?",
        "criteria": [
            "동작 원리를 전혀 설명하지 않는다: 역전파의 중요성이나 신경망에 대한 일반론뿐이다",
            "오차나 기울기가 뒤로 전달된다고 말하지만, 메커니즘이 없거나 틀렸다",
            "메커니즘은 대체로 맞지만, 중요한 오류나 누락이 하나 있다",
            "정확하고 완전하다: 순전파, 손실, 연쇄법칙을 통한 기울기 계산, 가중치 갱신",
        ],
    },
    "misc_is_optimizer": {
        "type": "noul",
        "instructions": "`answer`가 역전파 자체를 가중치를 갱신하는 최적화 알고리즘이라고 주장한다.",
        "criteria": {
            "true": "역전파를 최적화기나 가중치 갱신 알고리즘이라고 부르며 경사하강법과 혼동한다",
            "false": "역전파를 기울기 계산 방법으로 다루거나, 기울기 계산과 가중치 갱신을 별개 단계로 서술한다",
        },
    },
    "misc_error_copied": {
        "type": "noul",
        "instructions": "`answer`가 미분 계산 없이 같은 오차 값이 모든 층에 그대로 전달된다고 주장한다.",
    },
    "misc_reverse_inference": {
        "type": "noul",
        "instructions": "`answer`가 역전파를 신경망을 거꾸로 실행해 입력을 다시 계산하는 것으로 설명한다.",
    },
}

# V2와 구조가 같은지 확인 (같아야 EXPECTED_V2를 공유할 수 있음)
assert QUESTIONS_V2_KO.keys() == QUESTIONS_V2.keys()
for qid, q in QUESTIONS_V2.items():
    ko = QUESTIONS_V2_KO[qid]
    assert ko["type"] == q["type"], qid
    assert len(ko.get("criteria") or []) == len(q.get("criteria") or []), qid

In [ ]:
rows_ko = []
for name, text in ANSWERS.items():
    data, latency = call_jev(make_state(text), QUESTIONS_V2_KO, tag=f"run_v2_ko_{name}")
    for row in apply_gate(parse_answers(data)):
        rows_ko.append({"answer": name, "latency": latency, "input_tokens": data["usage"]["input_tokens"], **row})

df_ko = pd.DataFrame(rows_ko)
df_ko.to_csv(OUT_DIR / "run_v2_ko.csv", index=False)

display(df_ko.assign(pred=df_ko.pred.where(~df_ko.skipped, "skip")).pivot(index="answer", columns="question", values="pred"))
display(df_ko.pivot(index="answer", columns="question", values="conf"))

### 영어 vs 한국어: 라벨 기준
`high_conf_wrong`(자동 처리됐는데 라벨과 다른 칸)이 0이어야 한국어 질문을 그대로 쓸 수 있음.

In [ ]:
def label_cmp(d: pd.DataFrame) -> pd.DataFrame:
    exp = pd.DataFrame(
        [{"answer": a, "question": q, "expected": v} for a, qs in EXPECTED_V2.items() for q, v in qs.items()]
    )
    c = d.merge(exp, on=["answer", "question"])
    c = c[~c.skipped & c.expected.notna()].copy()
    c["correct"] = c.pred == c.expected
    return c

lang_summary = {}
for lang, d in [("en", df_v2), ("ko", df_ko)]:
    c = label_cmp(d)
    auto = is_auto(c, T_NOUL, T_OTHER)
    lang_summary[lang] = {
        "label_match": f"{c.correct.sum()}/{len(c)}",
        "auto_rate": round(auto.mean(), 2),
        "auto_accuracy": round(c[auto].correct.mean(), 2),
        "high_conf_wrong": int((auto & ~c.correct).sum()),
        "input_tokens_mean": int(d.groupby("answer").input_tokens.first().mean()),
    }
display(pd.DataFrame(lang_summary))

cmp_ko = label_cmp(df_ko)
display(cmp_ko[~cmp_ko.correct][["answer", "question", "raw", "pred", "expected", "conf"]].sort_values("conf", ascending=False))

### 영어 vs 한국어: 칸별 차이
판단이 바뀌었거나 확신도가 0.1 이상 차이 나는 칸. 안정성 섹션의 실행 간 편차(최대 약 0.07)보다 크면 언어 차이로 볼 수 있음.

In [ ]:
lang_diff = df_v2.merge(df_ko, on=["answer", "question"], suffixes=("_en", "_ko"))
lang_diff = lang_diff[~(lang_diff.skipped_en & lang_diff.skipped_ko)]
lang_diff["pred_changed"] = lang_diff.pred_en.astype(str) != lang_diff.pred_ko.astype(str)
lang_diff["conf_diff"] = (lang_diff.conf_ko - lang_diff.conf_en).round(2)

print(f"판단 변경: {lang_diff.pred_changed.sum()}/{len(lang_diff)}칸")
print(f"확신도 차이(ko - en): 평균 {lang_diff.conf_diff.mean():+.3f}, 절댓값 최대 {lang_diff.conf_diff.abs().max():.2f}")
display(
    lang_diff[lang_diff.pred_changed | (lang_diff.conf_diff.abs() >= 0.1)]
    [["answer", "question", "pred_en", "pred_ko", "conf_en", "conf_ko", "conf_diff"]]
)

### 한국어 안정성
영어 V2와 같은 방식으로 5회 반복.

In [ ]:
def summarize_stability(s: pd.DataFrame) -> pd.DataFrame:
    out = (
        s.assign(pred_str=s.pred.astype(str))
        .groupby(["answer", "question"])
        .agg(conf_mean=("conf", "mean"), conf_std=("conf", "std"), conf_min=("conf", "min"), n_pred=("pred_str", "nunique"))
        .round(3)
    )
    out["flipped"] = out.n_pred > 1
    return out

stab_ko_rows = []
for k in range(REPEATS):
    for name, text in ANSWERS.items():
        data, _ = call_jev(make_state(text), QUESTIONS_V2_KO, tag=f"stab_ko_{name}_{k}", save=False)
        for row in parse_answers(data):
            stab_ko_rows.append({"run": k, "answer": name, **row})

stab_ko = pd.DataFrame(stab_ko_rows)
stab_ko.to_csv(OUT_DIR / "stability_ko.csv", index=False)

stab_by_lang = {"en": summarize_stability(stab), "ko": summarize_stability(stab_ko)}
display(pd.DataFrame({
    lang: {
        "flipped": f"{s.flipped.sum()}/{len(s)}",
        "conf_std_median": s.conf_std.median(),
        "conf_std_max": s.conf_std.max(),
    }
    for lang, s in stab_by_lang.items()
}))

s_ko = stab_by_lang["ko"]
display(s_ko[s_ko.flipped | (s_ko.conf_std >= 0.05)])

## 관찰

> 문항 1개, 답안 6개, 36칸 기준. 수치는 방향 확인용이고 확정은 4주차 평가셋에서.
> `EXPECTED_V2`는 Claude 초안 라벨이며 Jev 결과를 본 뒤 매겼으므로 독립 라벨이 아님.

### 1. 응답 구조
- 문서와 일치. 질문 id가 `answers` 키로 그대로 반환
- 값은 소수 둘째 자리로 반올림 (0.99는 상한이 아님, 1.0과 0.0도 나옴)
- Noul은 `confidence` 없음 → `|2p-1|`로 정의
- Score `probabilities` 키는 문자열 `"0"`~`"3"`
- 입력 약 800 토큰/건(영어 질문 7개) → 약 $0.00003/건

### 2. 확신도 분포
- 명확한 판단(모범 답안, 무관 답안, 연쇄법칙 유무)은 0.9 이상
- 낮은 확신도는 사람이 애매하다고 본 칸과 겹침: `partial`의 accuracy·is_optimizer, `misconception`의 weight_update·is_optimizer, `verbose`의 weight_update
- `|2p-1|`은 Noul에 엄격 (p=0.11~0.17의 명백한 False도 0.66~0.78) → 타입별 임계값
- 안정성: 같은 입력 5회 반복 시 판단 변경 0~1/42, 확신도 표준편차 최대 0.07 미만. 판단이 바뀌는 칸은 p≈0.5(확신도 ≈ 0)뿐이라 어차피 에스컬레이션됨
- 고확신 오답: 0 (영어·한국어 질문 모두)
- 임계값 잠정값: `T_NOUL = 0.6` (p ≤ 0.2 또는 p ≥ 0.8), `T_OTHER = 0.8` → 자동 처리 86~94%, 자동 처리 정확도 100%

### 3. 지연
- 단건 0.16~0.30s
- 질문 1 / 4 / 16개 모두 중앙값 0.17~0.19s → 질문 수 영향 없음 (문서 주장 확인)
- 입력 토큰은 질문당 약 60개 증가

### 4. 한국어 품질
- 한국어 답안 + 영어 질문: 라벨 35~36/36
- 한국어 질문(`QUESTIONS_V2_KO`): 라벨 36/36, 고확신 오답 0, 자동 처리율 94%, 안정성 영어와 동등, 입력 토큰 +26%
- 영어 ↔ 한국어 질문 간 판단 변경 1/36 (p≈0.5 칸)
- **주의:** 한국어 질문이 애매한 칸에서 확신도가 더 높음 (`misconception` weight_update .02 → .64, `verbose` weight_update .44 → .78). 판단은 맞았지만 확신도 보정이 덜 됐을 가능성 → 평가셋에서 한국어 질문의 확신도 구간별 정확도 확인 필요

### 5. 모델 버전
- `jev-1.14.0`: 판단 변경 1/42 (p≈0.5 칸), 오개념 항목 확신도 상승 (`misconception` is_optimizer .56 → .84)
- 개발은 `jev-1.13.0` 고정, 평가셋에서 두 버전 비교 후 선택 (임계값은 버전별로 재조정)

### 질문 설계 교훈
- 오개념은 Choice가 아니라 오개념별 Noul (동시에 여러 개 존재 가능)
- 주제 관련성은 게이트 Noul로 분리하고 분기는 코드에서
- Score 수준 하나에 두 상황을 섞지 않기
- 질문 문구가 채점 의도와 다르면 라벨이 아니라 문구를 고친다

### 설계 반영 사항
- `Judgment` 필드: `question_id`, `type`, `raw`, `pred`, `conf`, `skipped`
- Noul 확신도: `|2p-1|`, 임계값은 Score/Choice와 별도
- Score 정수 변환: `argmax(probabilities)`, 연속값은 `raw`로 보존
- 질문 언어: 한국어 그대로 사용 (번역 단계 없음)
- 모델은 버전 고정 (alias는 새 버전 출시 시 응답이 바뀔 수 있음)